[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Intertangler/CB2330-exercises/blob/main/06_optimization/exercise.ipynb)

# Session 06 - Optimisation and Gradient Descent

In this exercise we walk down last session's score instead of sweeping it. Its score is last session's negative log-likelihood over 25 dishes, and it reaches 8.24 colonies per dish, as the sweep and the closed form did. The cost differs: 400 candidates scored blind against 13 steps, with the slope measured at a candidate placing the next.

I've written the data, last session's score and sweep, check cells and figures already.

**Deliverable:** this notebook, run, committed to `cb2330-portfolio`, with its figures rendered on the published page and a line under section 4's figure saying which run to report.

---

# Part A

The data are last session's colonies counted on 25 dishes, in colonies per dish, and `score` is their negative log-likelihood as a function of $\theta$ alone.

In [ ]:
import math
import matplotlib.pyplot as plt

# colonies counted on 25 dishes, in colonies per dish, as fitted last session
counts = [6, 9, 3, 9, 8, 11, 8, 13, 5, 9, 9, 6, 7, 14, 7, 7, 8, 7, 3, 10, 7, 15, 11, 7, 7]
N = len(counts)                                   # dishes
total_colonies = 0
for x in counts:
    total_colonies = total_colonies + x
mean = total_colonies / N                         # the analytical estimate, 8.24 colonies per dish
print(N, "dishes,", total_colonies, "colonies, sample mean", mean, "colonies per dish")


def factorial(x):
    """x! as a product, 0! = 1."""
    total = 1
    for k in range(1, x + 1):
        total = total * k
    return total


def poisson(x, theta):
    """p(x | theta) = theta^x e^(-theta) / x!  the probability of x colonies at rate theta."""
    return theta ** x * math.exp(-theta) / factorial(x)


def nll(counts, theta):
    """NLL(theta) = - the sum over dishes of log p(x_i | theta)."""
    total = 0.0
    for x in counts:
        total = total - math.log(poisson(x, theta))
    return total


def score(theta):
    """NLL(theta) for the 25 dishes, as a function of theta alone."""
    return nll(counts, theta)


Below is last session's sweep, 400 candidates scored in turn, and its estimate, kept for comparison.

In [ ]:
# the sweep of last session: 400 candidates, scored one by one
M = 400
candidates = [0.0] * M
scores = [0.0] * M
for j in range(M):
    candidates[j] = 0.05 * (j + 1)                # 0.05 to 20.00 colonies per dish
    scores[j] = score(candidates[j])


def position_of_min(values):
    """The index of the smallest entry: the position, not the value."""
    best = 0
    for j in range(len(values)):
        if values[j] < values[best]:
            best = j
    return best


theta_hat_sweep = candidates[position_of_min(scores)]
print("sweep:", M, "scores, theta_hat =", theta_hat_sweep, "colonies per dish, NLL =", round(score(theta_hat_sweep), 2))


## 1. Slope

This section is aimed at the slope as a measurement. A Poisson's slope has a closed form, which last session's page derived, and most models do not, so the page measures it from a pair of scores instead: a little above a candidate, a little below, subtract, and divide by their separation. We will measure it at $\theta = 4$, where the page reads $-26.5$, and watch it cross zero at the sample mean.

$$\frac{d\,\mathrm{NLL}}{d\theta} \approx \frac{\mathrm{NLL}(\theta + h) - \mathrm{NLL}(\theta - h)}{2h}$$

`slope_at` takes a score, a candidate $\theta$ and an offset $h$, and returns the slope there. A check compares it with last session's closed form, $N - \frac{1}{\theta}\sum_i x_i$, and a figure plots slope against $\theta$ with its zero crossing marked.

**Fill the gap to measure the slope.** `slope_at` receives `score`, `theta` and `h` and returns the formula above: `score(theta + h)` minus `score(theta - h)`, over `2 * h`. Dividing by `h` alone gives twice the slope, and the check catches it.

In [ ]:
def slope_at(score, theta, h):
    """dNLL/dtheta at theta, by central difference: score h above and h below, over 2h."""
    # ⭐⭐⭐ Your code here ⭐⭐⭐
    pass


In [ ]:
analytic_at_4 = N - total_colonies / 4.0          # the page's closed form, N - (1/theta) sum x_i
assert abs(analytic_at_4 - (-26.5)) < 1e-12
assert abs(slope_at(score, 4.0, 0.01) - analytic_at_4) < 0.001, "the central difference lands on the closed form"
assert abs(slope_at(score, mean, 0.01)) < 0.001, "flat at the sample mean"
assert slope_at(score, 12.0, 0.01) > 0, "positive past the minimum"
print("ok  slope at 4 =", round(slope_at(score, 4.0, 0.01), 3), "per colony per dish;  closed form", analytic_at_4)
print("    slope at the mean =", round(slope_at(score, mean, 0.01), 6))


In [ ]:
view = [0.0] * 121
slopes = [0.0] * 121
for j in range(121):
    view[j] = 2.0 + 0.1 * j                       # 2 to 14 colonies per dish
    slopes[j] = slope_at(score, view[j], 0.01)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(view, slopes, color="black")
ax.plot([2.0, 14.0], [0.0, 0.0], color="grey", linewidth=0.6)
ax.plot([mean], [0.0], marker="o", color="red", label="the sample mean, slope 0")
ax.set_xlabel("theta, colonies per dish")
ax.set_ylabel("dNLL/dtheta, per colony per dish")
ax.set_title("the slope of the score, measured by central difference")
ax.legend()
plt.show()


## 2. Step

A slope's sign says which way the score falls and its size says how steeply, so a step against it, by a multiple $\alpha$ we choose, moves us down. The minus sign carries the concept: at 4 the slope is negative, so the score falls rightward and the step goes right. We will take the page's first step, from 4 to 6.650, and see the score drop from 104.4 to 65.9.

$$\theta_{\text{new}} = \theta_{\text{old}} - \alpha\,\frac{d\,\mathrm{NLL}}{d\theta}$$

This cell takes a step from $\theta = 4$ with $\alpha = 0.1$, and the check cell tests where it lands.

**Fill the gap to take a step.** `theta_old`, `alpha` and `slope` are set. Write `theta_new` as `theta_old` minus `alpha` times `slope`.

In [ ]:
theta_old = 4.0                                   # colonies per dish, the page's start
alpha = 0.1                                       # (colonies per dish)^2
slope = slope_at(score, theta_old, 0.01)
theta_new = ⭐⭐⭐ Your code here ⭐⭐⭐


In [ ]:
assert abs(theta_new - 6.65) < 0.001, "one step from 4 at alpha = 0.1 lands on the page's 6.650"
assert theta_new > theta_old, "the slope is negative, so the step goes up in theta"
assert score(theta_new) < score(theta_old), "and the score fell"
print("ok  theta_new =", round(theta_new, 3), "colonies per dish;  score", round(score(theta_old), 2), "->", round(score(theta_new), 2))


## 3. Walk

Here we turn section 2's step into a loop, and that loop is the search: measure, test for flat, step, repeat. Converged means a flat slope at the stop, inside a tolerance we chose, and does not say whether the model fits. We will run the page's walk from 4 at $\alpha = 0.1$ and reach 8.224 in 13 steps, 26 scores against the sweep's 400.

$$\theta_{\text{new}} = \theta_{\text{old}} - \alpha\,\frac{d\,\mathrm{NLL}}{d\theta}, \quad \text{repeated}; \qquad \text{stop when } \left|\frac{d\,\mathrm{NLL}}{d\theta}\right| < \varepsilon$$

`descend` takes a score, a starting candidate and a step size, with offset $h$, tolerance $\varepsilon$ and a step budget as defaults, and returns the candidates visited as a list. The check cell tests the stop against the page's table and sweep, and counts scores spent.

**Fill the gap to write the walk.** `descend` receives `score`, `theta`, `alpha`, `h`, `tol` and `max_steps`, and `path` is set up holding the start. Write a loop over `range(max_steps)` that measures `slope` with `slope_at`, returns `path` as soon as `abs(slope) < tol`, and otherwise moves `theta` by section 2's step and records it with `path.append(theta)`. `return path` after the loop is for a walk that spends its budget.

In [ ]:
def descend(score, theta, alpha, h=0.01, tol=0.05, max_steps=40):
    """Walk against the slope from theta. Returns the candidates visited, the last one being the estimate."""
    path = [theta]
    # ⭐⭐⭐ Your code here ⭐⭐⭐
    return path


In [ ]:
path = descend(score, 4.0, 0.1)
steps = len(path) - 1
theta_hat_walk = path[-1]
assert steps == 13, "the page's run converges at step 13"
assert abs(theta_hat_walk - 8.224) < 0.001, "and lands on the page's 8.224"
assert abs(slope_at(score, theta_hat_walk, 0.01)) < 0.05, "the slope at the stop is inside the tolerance"
assert abs(theta_hat_walk - theta_hat_sweep) <= 0.05, "within a grid step of the sweep"
assert abs(theta_hat_walk - mean) < 0.02, "and close to the closed form"
print("ok  walk:", steps, "steps,", 2 * steps, "scores, theta_hat =", round(theta_hat_walk, 3), "colonies per dish, NLL =", round(score(theta_hat_walk), 2))
print("    sweep:", M, "scores, theta_hat =", theta_hat_sweep)


## 4. Step size

A step size is ours to choose, and the page's runs show the price of choosing badly: too small crawls and spends its budget short of the minimum, too large jumps over it and back. We will run them from 4 and diagnose from slope and step count rather than from the estimate, since a run that spent its budget returns a number either way.

`show` prints the page's table for a walk: step, candidate, score and slope. Checks diagnose the runs from their paths, and a figure draws them on the score.

**Under the figure, say in a line which run to report, and what slope and step count say about the others.**

In [ ]:
def show(path):
    """The page's table: step, theta, NLL, slope. The first 5 rows and the last 3."""
    print("step  theta    NLL     slope")
    for i in range(len(path)):
        if i < 5 or i >= len(path) - 3:
            print(i, "   ", round(path[i], 3), "  ", round(score(path[i]), 2), "  ", round(slope_at(score, path[i], 0.01), 2))
        if i == 5 and len(path) > 8:
            print("...")


path_small = descend(score, 4.0, 0.002)
path_mid = descend(score, 4.0, 0.1)
path_large = descend(score, 4.0, 0.7)
print("alpha = 0.002")
show(path_small)
print("\nalpha = 0.1")
show(path_mid)
print("\nalpha = 0.7")
show(path_large)


In [ ]:
assert len(path_small) - 1 == 40, "the small step spends the whole budget"
assert slope_at(score, path_small[-1], 0.01) < -0.05, "and stops on a slope still pointing down"
assert len(path_mid) - 1 == 13, "the middle step converges inside the budget"
assert len(path_large) - 1 == 40, "the large step spends the whole budget"
assert (path_large[-1] - mean) * (path_large[-2] - mean) < 0, "and its last two candidates sit on opposite sides of the minimum"
print("ok  0.002: budget spent, slope", round(slope_at(score, path_small[-1], 0.01), 2), " still descending")
print("    0.1:   converged in", len(path_mid) - 1, "steps")
print("    0.7:   budget spent, last candidates", round(path_large[-2], 3), "and", round(path_large[-1], 3), " on opposite sides of", mean)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(candidates, scores, color="black", linewidth=0.8)
heights = [0.0] * len(path_small)
for i in range(len(path_small)):
    heights[i] = score(path_small[i])
ax.plot(path_small, heights, marker="o", markersize=3, color="blue", label="alpha = 0.002, crawls")
heights = [0.0] * len(path_mid)
for i in range(len(path_mid)):
    heights[i] = score(path_mid[i])
ax.plot(path_mid, heights, marker="o", markersize=3, color="green", label="alpha = 0.1, settles")
heights = [0.0] * len(path_large)
for i in range(len(path_large)):
    heights[i] = score(path_large[i])
ax.plot(path_large, heights, marker="o", markersize=3, color="red", label="alpha = 0.7, oscillates")
ax.set_xlim(2, 14)
ax.set_ylim(55, 130)
ax.set_xlabel("theta, colonies per dish")
ax.set_ylabel("NLL(theta), dimensionless")
ax.set_title("three walks on the score of 25 dishes, from theta = 4")
ax.legend()
plt.show()


## 5. Adaptive step

In the page's adaptive rule the score chooses the step size. A step is proposed and scored before it's accepted: a rise means it was too long, so we stay put and halve $\alpha$, and a fall means it was safe, so we take it and lengthen $\alpha$ by a fifth. We will start from step sizes that crawled and oscillated in section 4 and watch this rule repair them.

$$\theta_{\text{try}} = \theta_{\text{old}} - \alpha_{\text{old}}\,\frac{d\,\mathrm{NLL}}{d\theta}, \qquad (\theta_{\text{new}}, \alpha_{\text{new}}) = \begin{cases} (\theta_{\text{old}},\ \tfrac{1}{2}\alpha_{\text{old}}) & \mathrm{NLL}(\theta_{\text{try}}) > \mathrm{NLL}(\theta_{\text{old}}) \\ (\theta_{\text{try}},\ 1.2\,\alpha_{\text{old}}) & \text{otherwise} \end{cases}$$

`descend_adaptive` takes the same arguments as `descend` and returns the same list. The check cell tests that it converges inside the budget from either start, and a figure draws fixed and adaptive walks from 0.7 on the score.

**Fill the gap to let the score choose the step.** Loop, slope and stopping test are given. Write `theta_try` as section 2's step, then an `if` on `score(theta_try) > score(theta)` that halves `alpha` and leaves `theta` where it is, and an `else` that sets `theta` to `theta_try` and multiplies `alpha` by 1.2. `else` runs whenever the `if` above it did not.

In [ ]:
def descend_adaptive(score, theta, alpha, h=0.01, tol=0.05, max_steps=40):
    """The walk of descend, with the step size halved on a rise and lengthened by 1.2 on a fall."""
    path = [theta]
    for step in range(max_steps):
        slope = slope_at(score, theta, h)
        if abs(slope) < tol:
            return path
        # ⭐⭐⭐ Your code here ⭐⭐⭐
        path.append(theta)
    return path


In [ ]:
path_adaptive_large = descend_adaptive(score, 4.0, 0.7)
path_adaptive_small = descend_adaptive(score, 4.0, 0.002)
assert len(path_adaptive_large) - 1 < 40, "from 0.7 the adaptive walk converges inside the budget"
assert abs(path_adaptive_large[-1] - mean) < 0.02
assert len(path_adaptive_small) - 1 < 40, "and from 0.002 as well"
assert abs(path_adaptive_small[-1] - mean) < 0.02
assert path_adaptive_large[1] == path_adaptive_large[0], "the first proposal from 0.7 is refused, so the candidate stays put"
print("ok  adaptive from 0.7:  ", len(path_adaptive_large) - 1, "steps, theta_hat =", round(path_adaptive_large[-1], 3))
print("    adaptive from 0.002:", len(path_adaptive_small) - 1, "steps, theta_hat =", round(path_adaptive_small[-1], 3))
print("    plain from 0.7 and 0.002: 40 steps each, neither converged")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(candidates, scores, color="black", linewidth=0.8)
heights = [0.0] * len(path_large)
for i in range(len(path_large)):
    heights[i] = score(path_large[i])
ax.plot(path_large, heights, marker="o", markersize=3, color="red", label="alpha = 0.7, fixed")
heights = [0.0] * len(path_adaptive_large)
for i in range(len(path_adaptive_large)):
    heights[i] = score(path_adaptive_large[i])
ax.plot(path_adaptive_large, heights, marker="o", markersize=4, color="purple", label="alpha = 0.7 to start, adaptive")
ax.set_xlim(2, 14)
ax.set_ylim(55, 130)
ax.set_xlabel("theta, colonies per dish")
ax.set_ylabel("NLL(theta), dimensionless")
ax.set_title("the same start and step size, with and without the adaptive rule")
ax.legend()
plt.show()


---

# Part B

In Part B the score stops being a curve over $\theta$. On a pair of parameters the gradient has a component per parameter and a walk steps along $\mu$ and $\sigma$ at once, and on a score with a second dip a walk settles in whichever dip it started nearest, which a sweep does not do.

Part B is optional and done outside the session, and a route is enough.

- **$\mu$ and $\sigma$ together.** Below, already written: a walk on last session's normal, a slope per parameter with the other held still, 100 scores against 27,391 pairs on last session's grid.
- **A local minimum.** Also below: a rhythm's period, walked from several starting periods, and some runs stop in the wrong dip. The score at the stop separates them.
- **Map the step size.** Section 3's walk across step sizes from 0.001 to 1, classified as crawling, settling or oscillating. For these counts the boundary between settling and oscillating sits near 0.66, bisection finds it to a second decimal, and how fast the slope itself changes at the minimum, the score's second derivative $\sum_i x_i / \hat{\theta}^2 = 3.03$, predicts it as $2 / 3.03$.
- **Shrink the step, or shake it.** The page's schedule, $\alpha_{\text{new}} = 0.97\,\alpha_{\text{old}}$, run from 0.7 where a fixed step oscillates. And the page's random step, $\delta\,(2r - 1)$ with $r$ from the sampling session's generator, added to a walk on the rhythm score from 14 hours, where a plain walk is stuck. The amplitude at which runs begin to reach 24 hours compares with the gap between the dips.
- **Hand it to a library.** `scipy.optimize.minimize` on `score` and on `nll_period` from the same starts, with estimates, evaluation counts and stopping points set beside the loop written here.
- **Break the offset.** `slope_at` with $h = 10^{-12}$, and its slope read against the pair of scores it was taken from.

## B1. $\mu$ and $\sigma$ together

$$\frac{\partial\,\mathrm{NLL}}{\partial \mu} \approx \frac{\mathrm{NLL}(\mu + h, \sigma) - \mathrm{NLL}(\mu - h, \sigma)}{2h}, \qquad \mu_{\text{new}} = \mu_{\text{old}} - \alpha\,\frac{\partial\,\mathrm{NLL}}{\partial \mu}, \quad \sigma_{\text{new}} = \sigma_{\text{old}} - \alpha\,\frac{\partial\,\mathrm{NLL}}{\partial \sigma}$$

`slope_mu` and `slope_sigma` are the gradient's components along $\mu$ and along $\sigma$, taken with the other parameter held still. The loop steps $\mu$ and $\sigma$ together and stops once slopes along $\mu$ and $\sigma$ are inside tolerance. The check cell compares $\hat{\mu}$ and $\hat{\sigma}$ with sample mean and standard deviation, and the figure draws the walk on the score's contours.

In [ ]:
# the diameter of one colony per dish, measured with a ruler, in mm, as fitted last session
diameters = [2.0, 2.0, 2.6, 1.6, 2.3, 1.6, 2.8, 2.5, 2.9, 2.2, 2.5, 2.3, 2.1,
             2.5, 2.0, 2.3, 2.6, 2.4, 2.3, 3.2, 2.4, 2.2, 2.5, 2.2, 2.3]


def normal(x, mu, sigma):
    """f(x | mu, sigma)  the density of a diameter x, in 1/mm."""
    return math.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / (sigma * (2 * math.pi) ** 0.5)


def nll_normal(diameters, mu, sigma):
    """NLL(mu, sigma) = - the sum over dishes of log f(x_i | mu, sigma)."""
    total = 0.0
    for x in diameters:
        total = total - math.log(normal(x, mu, sigma))
    return total


def slope_mu(mu, sigma, h):
    """The partial slope along mu, with sigma held still."""
    return (nll_normal(diameters, mu + h, sigma) - nll_normal(diameters, mu - h, sigma)) / (2 * h)


def slope_sigma(mu, sigma, h):
    """The partial slope along sigma, with mu held still."""
    return (nll_normal(diameters, mu, sigma + h) - nll_normal(diameters, mu, sigma - h)) / (2 * h)


mu = 2.0                                          # mm, the start
sigma = 1.0                                       # mm
alpha_2 = 0.002                                   # mm^2
path_mu = [mu]
path_sigma = [sigma]
for step in range(400):
    s_mu = slope_mu(mu, sigma, 0.001)
    s_sigma = slope_sigma(mu, sigma, 0.001)
    if abs(s_mu) < 0.05 and abs(s_sigma) < 0.05:
        break                                     # ends the loop early, as return did inside descend
    mu = mu - alpha_2 * s_mu
    sigma = sigma - alpha_2 * s_sigma
    path_mu.append(mu)
    path_sigma.append(sigma)
steps_2 = len(path_mu) - 1


In [ ]:
mean_d = 0.0
for x in diameters:
    mean_d = mean_d + x / len(diameters)
variance_d = 0.0
for x in diameters:
    variance_d = variance_d + (x - mean_d) ** 2 / len(diameters)
sd_d = variance_d ** 0.5
assert steps_2 < 400, "converged inside the budget"
assert abs(mu - mean_d) < 0.005, "mu_hat lands on the sample mean"
assert abs(sigma - sd_d) < 0.005, "sigma_hat lands on the standard deviation, with 1/N"
print("ok  two-parameter walk:", steps_2, "steps,", 4 * steps_2, "scores;  mu_hat =", round(mu, 3), "mm, sigma_hat =", round(sigma, 3), "mm")
print("    sample mean", round(mean_d, 3), "mm, standard deviation", round(sd_d, 3), "mm;  the grid of last session scored 27391 pairs")


In [ ]:
view_mu = [0.0] * 60
view_sigma = [0.0] * 60
surface = []                                      # a row per sigma, a column per mu
for b in range(60):
    view_sigma[b] = 0.15 + 0.015 * b
    row = [0.0] * 60
    for a in range(60):
        view_mu[a] = 1.8 + 0.01 * a
        row[a] = nll_normal(diameters, view_mu[a], view_sigma[b])
    surface.append(row)
levels = [0.0] * 10
for k in range(10):
    levels[k] = nll_normal(diameters, mu, sigma) + 0.5 * (k + 1) ** 2
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.contour(view_mu, view_sigma, surface, levels=levels, colors="black", linewidths=0.6)
ax.plot(path_mu, path_sigma, marker="o", markersize=3, color="purple", label="the walk, from (2.0, 1.0)")
ax.plot([mu], [sigma], marker="o", color="red", label="the estimate")
ax.set_xlabel("mu, mean diameter in mm")
ax.set_ylabel("sigma, standard deviation in mm")
ax.set_title("NLL(mu, sigma) over both parameters, and the walk down it")
ax.legend()
plt.show()


## B2. A local minimum

$$\lambda(t) = 20\,\bigl(1 + 0.8\cos(2\pi t / T)\bigr), \qquad \mathrm{NLL}(T) = -\sum_{i=1}^{48} \ln p\bigl(x_i \mid \lambda(t_i)\bigr)$$

`photons` holds 48 hourly counts from a luciferase reporter, a gene whose product glows so that photons per frame follow its activity, `rate` is a Poisson mean rising and falling with the hour, and `nll_period` scores a candidate period $T$. This cell walks `descend` from several starting periods. The check cell tests where the runs stopped, and a figure draws $\mathrm{NLL}(T)$ against $T$ with starts and stops marked.

In [ ]:
# photons counted per hour-long frame from a luciferase reporter, 48 frames, a circadian rhythm
photons = [37, 33, 37, 34, 16, 30, 22, 12, 15, 9, 6, 4, 2, 6, 8, 16, 14, 21, 20, 36, 19, 37, 47, 29,
           27, 31, 33, 23, 29, 27, 17, 21, 15, 10, 7, 3, 2, 3, 12, 9, 9, 11, 19, 26, 29, 23, 32, 32]
hours = [0.0] * 48
for i in range(48):
    hours[i] = 1.0 * i                            # the frame's start, in hours


def rate(t, T):
    """The mean photons per frame at hour t for a rhythm of period T, mean 20 and swing 0.8."""
    return 20 * (1 + 0.8 * math.cos(2 * math.pi * t / T))


def nll_period(T):
    """NLL(T) = - the sum over frames of log p(x_i | rate(t_i, T)), a Poisson per frame."""
    total = 0.0
    for i in range(48):
        total = total - math.log(poisson(photons[i], rate(hours[i], T)))
    return total


starts = [10.0, 14.0, 20.0, 28.0]                 # hours
ends = [0.0] * 4
for k in range(4):
    path_T = descend(nll_period, starts[k], 0.02, h=0.01, tol=0.05, max_steps=200)
    ends[k] = path_T[-1]
    print("start", starts[k], "h  ->  T_hat =", round(ends[k], 2), "h after", len(path_T) - 1, "steps, NLL =", round(nll_period(ends[k]), 1))


In [ ]:
assert abs(ends[0] - 14.68) < 0.05 and abs(ends[1] - 14.68) < 0.05, "two starts settle in the shallow dip"
assert abs(ends[2] - 24.0) < 0.05 and abs(ends[3] - 24.0) < 0.05, "two starts settle in the deep one"
assert nll_period(ends[2]) < nll_period(ends[0]) - 200, "and the deep one scores far better"
print("ok  the dips: T =", round(ends[0], 2), "h at NLL", round(nll_period(ends[0]), 1), " and T =", round(ends[2], 2), "h at NLL", round(nll_period(ends[2]), 1))


In [ ]:
periods = [0.0] * 137
scores_T = [0.0] * 137
for j in range(137):
    periods[j] = 6.0 + 0.25 * j                   # 6 to 40 hours
    scores_T[j] = nll_period(periods[j])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(periods, scores_T, color="black")
for k in range(4):
    ax.plot([starts[k]], [nll_period(starts[k])], marker="o", color="blue")
    ax.plot([ends[k]], [nll_period(ends[k])], marker="o", color="red")
ax.plot([], [], marker="o", color="blue", linestyle="none", label="a start")
ax.plot([], [], marker="o", color="red", linestyle="none", label="where the walk stopped")
ax.set_xlabel("T, period in hours")
ax.set_ylabel("NLL(T), dimensionless")
ax.set_title("the score over the period, 48 frames, and four walks")
ax.legend()
plt.show()
